# Employee Insights Pipeline — A Spark-Based Big Data Project

**Objective:** Build an end-to-end ETL (Extract, Transform, Load) pipeline using Apache Spark 
to clean raw employee data and generate multiple business-relevant summary views, 
in the style of a data dashboard.

**Dataset:** Synthetic employee records (1500+ rows) containing department, salary, 
age, city, employment status, and years of experience — including intentional 
data quality issues (nulls, duplicates) to practice real-world cleaning.

**Tools used:** PySpark (DataFrame API + Spark SQL), pandas (for final CSV export)

## 1. Setup and Data Extraction

Initializing the Spark session and loading the raw employee CSV into a DataFrame. 
This is the **Extract** stage of the ETL pipeline — Spark reads the file and 
prepares it for distributed processing.

In [27]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("EmployeesInsightPipeline").getOrCreate()

raw_df = spark.read.csv(r"C:\Users\yasmi\Downloads\employees.csv" , header = True , inferSchema = True )

raw_df.show()

+------+-----------+---------+---+---------+--------+----------------+
|emp_id| department|   salary|age|     city|  status|years_experience|
+------+-----------+---------+---+---------+--------+----------------+
|     1|         HR|  65479.1| 31|   Mumbai|  Active|              26|
|     2|      Sales| 88429.47| 24|  Chennai|Inactive|               2|
|     3|    Finance| 95681.71| 36|    Kochi|  Active|              18|
|     4|  Marketing| 94583.11| 30|  Chennai|  Active|              17|
|     5|    Finance|  47590.8| 58|    Delhi|Inactive|               3|
|     6|         HR| 50744.84| 56|Bangalore|Inactive|              18|
|     7|      Sales| 47470.84| 37|    Kochi|Inactive|              16|
|     8|      Sales|101119.88| 26|    Kochi|Inactive|               5|
|     9|    Support| 29900.91| 26|    Kochi|Inactive|              10|
|    10|  Marketing| 82025.67| 27|  Chennai|On Leave|              21|
|    11|  Marketing|     NULL| 50|  Chennai|On Leave|              11|
|    1

## 2. Data Quality Check — Null Values

Before cleaning, we check how many missing values exist in each column. 
This tells us exactly what needs fixing before any analysis is trustworthy.

**Insight:** The raw dataset had 30 missing values in `department`, 70 in `salary`, 
and 41 in `city` — no missing values in `emp_id`, `age`, `status`, or 
`years_experience`. This confirms the data quality issues were concentrated 
in exactly the three columns we planned to clean.

In [28]:
raw_df.select(
    [
        F.sum(
            F.when(
                F.col(c).isNull() , 1
            ).otherwise(0)
        ).alias(c)

        for c in raw_df.columns
    ]
).show()

+------+----------+------+---+----+------+----------------+
|emp_id|department|salary|age|city|status|years_experience|
+------+----------+------+---+----+------+----------------+
|     0|        30|    70|  0|  41|     0|               0|
+------+----------+------+---+----+------+----------------+



## 3. Data Cleaning

Applying three cleaning steps:
- Removing duplicate rows
- Filling missing `salary` values with the dataset's overall average
- Filling missing `city` and `department` values with `"Unknown"`

This ensures downstream aggregations (counts, averages) aren't skewed by 
duplicate records or silently distorted by missing data.

**Insight:** After cleaning, all columns show zero null values — confirming 
that `dropDuplicates()` and `fillna()` successfully resolved every missing 
value across `department`, `salary`, and `city`.

In [29]:
avg_salary = raw_df.select(F.avg("salary")).collect()[0][0]

clean_df = raw_df.drop_duplicates().fillna({"salary" : avg_salary , "city" : "Unknown" , "department" : "Unknown"})

clean_df.select(
    [
        F.sum(
            F.when(
                F.col(c).isNull() , 1
            ).otherwise(0)
        ).alias(c)

        for c in clean_df.columns
    ]
).show()

+------+----------+------+---+----+------+----------------+
|emp_id|department|salary|age|city|status|years_experience|
+------+----------+------+---+----+------+----------------+
|     0|         0|     0|  0|   0|     0|               0|
+------+----------+------+---+----+------+----------------+



## 4. Feature Enrichment — Seniority Level

Adding a derived column, `seniority_level`, categorizing employees as 
Senior (10+ years), Mid (4-9 years), or Junior (under 4 years) based on 
`years_experience`. This enables more meaningful grouping in later analysis.

In [37]:
enriched_df = clean_df.withColumn(
    "seniority_level" ,
    F.when(clean_df.years_experience >= 10 , "Senior")
    .when((clean_df.years_experience >= 4) & (clean_df.years_experience <= 9) , "Mid")
    .otherwise("Junior")
)

enriched_df.cache()

enriched_df.show()

+------+----------+-----------------+---+---------+--------+----------------+---------------+
|emp_id|department|           salary|age|     city|  status|years_experience|seniority_level|
+------+----------+-----------------+---+---------+--------+----------------+---------------+
|    98|   Finance|        137270.93| 42|   Mumbai|  Active|              29|         Senior|
|   169|        HR|         45723.88| 27|    Kochi|  Active|              20|         Senior|
|   220|   Support|        123280.28| 39|Bangalore|Inactive|               8|            Mid|
|   605|        HR|        106991.53| 46|Bangalore|  Active|               7|            Mid|
|   640|   Finance|        129690.97| 24|  Chennai|On Leave|              12|         Senior|
|   949|   Finance|         84560.97| 51|  Chennai|  Active|              12|         Senior|
|  1027|   Support|        113010.81| 33|    Delhi|Inactive|              17|         Senior|
|  1301|        HR|        110006.23| 33|    Kochi|Inactive|

## 5. Insight 1: Headcount by Department

**Question:** How many employees work in each department?

**Insight:** Finance has the largest headcount at 270 employees, closely 
followed by HR (258) and Engineering (248). Support has the smallest 
headcount among named departments at 216. The 29 employees labeled 
"Unknown" represent department values that were missing in the raw data 
and filled during cleaning — about 1.9% of the total workforce (1500 employees).

In [35]:
# View 1: Headcount by department

view_headcount = clean_df.groupBy("department").count().withColumnRenamed("count" , "headcount")

view_headcount.show()

+-----------+---------+
| department|headcount|
+-----------+---------+
|      Sales|      232|
|Engineering|      248|
|         HR|      258|
|    Finance|      270|
|    Unknown|       29|
|  Marketing|      247|
|    Support|      216|
+-----------+---------+



In [33]:
#  Average salary per city.

clean_df.groupBy("city").agg(F.avg("salary")).show()

+---------+-----------------+
|     city|      avg(salary)|
+---------+-----------------+
|Bangalore|87914.28468111077|
|    Kochi|84878.79983683603|
|  Chennai| 87284.6775885447|
|   Mumbai|90194.95812546049|
|  Unknown|88731.82156164384|
|    Delhi|83497.84954531245|
+---------+-----------------+



**Insight:** Mumbai has the highest average salary at ₹90,194.96, while Delhi 
has the lowest at ₹83,497.85 — a gap of roughly ₹6,697. The differences 
across cities are relatively modest (all within about 8% of each other), 
suggesting salary in this dataset is not heavily location-dependent, unlike 
the sharper variation seen across departments and seniority levels.

In [34]:
# Finding employees with years_experience > 15 AND status == "Active".

enriched_df.filter(
    (enriched_df.years_experience > 15) & (enriched_df.status == "Active")
).show()

+------+-----------+---------+---+---------+------+----------------+---------------+
|emp_id| department|   salary|age|     city|status|years_experience|seniority_level|
+------+-----------+---------+---+---------+------+----------------+---------------+
|    98|    Finance|137270.93| 42|   Mumbai|Active|              29|         Senior|
|   169|         HR| 45723.88| 27|    Kochi|Active|              20|         Senior|
|   784|    Support| 67070.53| 28|    Delhi|Active|              22|         Senior|
|  1198|Engineering| 74394.71| 45|  Chennai|Active|              16|         Senior|
|  1054|Engineering|135783.88| 43|    Delhi|Active|              26|         Senior|
|  1241|      Sales| 51785.87| 25|Bangalore|Active|              21|         Senior|
|   933|  Marketing|138799.81| 23|  Unknown|Active|              30|         Senior|
|   769|    Finance| 43889.28| 36|   Mumbai|Active|              17|         Senior|
|   701|    Support|112960.01| 45|  Chennai|Active|              

**Insight:** A substantial number of highly experienced employees (15+ years) 
remain Active across every department — all sampled rows here are classified 
as "Senior," which is expected since anyone with 15+ years experience exceeds 
the 10-year Senior threshold by definition. This filter is most useful as a 
"retain and recognize" list — identifying your most tenured, currently active 
talent, such as employee #1385 in Marketing with 27 years of experience 
earning ₹148,254.19, one of the highest salaries in the entire dataset.

## 6. Insight 2: Average Salary by Department and Seniority Level

**Question:** How does average salary vary across departments and seniority levels?

**Insight:** Somewhat surprisingly, HR (Mid) shows the highest average 
salary of any department-seniority combination at ₹96,041.55 — even higher 
than any Senior-level average across other departments. The lowest average 
salary belongs to Support (Junior) at ₹73,974.37 — a gap of roughly ₹22,067 
between the highest and lowest paid group. This pattern suggests salary in 
this dataset isn't strictly tied to seniority level alone — department also 
plays a significant role, and the relationship between seniority and pay 
isn't perfectly linear (e.g., HR Mid outpaces HR Senior).

In [16]:
# View 2: Average salary by department and seniority level

view_avg_salary = enriched_df.groupBy("department" , "seniority_level").agg(F.avg("salary").alias("avg_salary"))

view_avg_salary.show()

+-----------+---------------+-----------------+
| department|seniority_level|       avg_salary|
+-----------+---------------+-----------------+
|    Support|         Senior|86699.70172602736|
|    Support|            Mid|84426.76925925924|
|         HR|         Senior|81840.92155300954|
|    Unknown|         Senior| 92021.3257142857|
|         HR|            Mid|96041.55200140497|
|    Finance|         Senior|88096.76586012973|
|  Marketing|         Senior|87375.64721952932|
|      Sales|         Senior|86466.26786127963|
|  Marketing|         Junior|90658.48894672754|
|    Support|         Junior|73974.36716394166|
|Engineering|         Senior|89382.90579435346|
|    Finance|            Mid| 86908.0740376712|
|      Sales|            Mid|85766.74600144196|
|Engineering|            Mid|91214.48796004565|
|      Sales|         Junior|88485.34483409436|
|Engineering|         Junior| 91431.0708371385|
|  Marketing|            Mid|86631.67800249069|
|         HR|         Junior|87052.49592

## 7. Insight 3: Employment Status Breakdown by Department

**Question:** How many Active, Inactive, and On Leave employees exist 
in each department?

**Insight:** Employment status is fairly evenly distributed across departments, 
with no department showing extreme concentration in any one status. HR has 
the highest proportion of Active employees (101 of 258, ~39.1%), while 
Finance and Support show the most balanced three-way split between Active, 
Inactive, and On Leave (roughly 32-34% each). This suggests employee status 
is not strongly tied to department — attrition and leave patterns appear 
fairly uniform across the organization.

In [17]:
# View 3: Status breakdown by department

view_status = enriched_df.groupby("department" , "status").count().withColumnRenamed("count" , "status_count")

view_status.show()

+-----------+--------+------------+
| department|  status|status_count|
+-----------+--------+------------+
|    Finance|  Active|          89|
|         HR|  Active|         101|
|  Marketing|Inactive|          85|
|    Unknown|On Leave|          11|
|    Finance|On Leave|          93|
|    Support|Inactive|          79|
|         HR|Inactive|          73|
|      Sales|Inactive|          69|
|    Support|On Leave|          68|
|    Support|  Active|          69|
|  Marketing|On Leave|          73|
|Engineering|  Active|          92|
|  Marketing|  Active|          89|
|    Finance|Inactive|          88|
|      Sales|On Leave|          86|
|Engineering|On Leave|          93|
|      Sales|  Active|          77|
|         HR|On Leave|          84|
|Engineering|Inactive|          63|
|    Unknown|Inactive|          10|
+-----------+--------+------------+
only showing top 20 rows


## 8. Insight 4: Top-Paying Department

**Question:** Which single department has the highest average salary overall?

**Insight:** Engineering is the top-paying department overall, with an 
average salary of ₹90,034.72 across all seniority levels combined — 
consistent with it also having a relatively high headcount (248 employees), 
suggesting the organization invests significantly in its engineering workforce.

In [18]:
# View 4: Top-paying department

view_top_dept = enriched_df.groupby("department").agg(F.avg("salary").alias("avg_salary")).orderBy(F.desc("avg_salary")).limit(1)

view_top_dept.show()

+-----------+----------------+
| department|      avg_salary|
+-----------+----------------+
|Engineering|90034.7198487627|
+-----------+----------------+



## 9. Insight 5: Overall Seniority Level Distribution

**Question:** What proportion of the entire workforce falls into each 
seniority tier?

**Insight:** The workforce is heavily weighted toward Senior employees, who 
make up 64.4% (966 of 1500) of the total headcount — substantially more than 
Mid-level (19.8%, 297 employees) and Junior (15.8%, 237 employees) combined. 
This indicates a fairly experienced, senior-heavy workforce overall, rather 
than a typical pyramid structure with more junior staff at the base.

In [36]:
# View 5: Seniority level distribution (overall)

view_seniority_dist = enriched_df.groupBy("seniority_level").count().withColumnRenamed("count" , "employee_count")

view_seniority_dist.show()

+---------------+--------------+
|seniority_level|employee_count|
+---------------+--------------+
|         Senior|           966|
|            Mid|           297|
|         Junior|           237|
+---------------+--------------+



## 10. Load — Saving Summary Views

Each summary view is exported as a separate CSV file into the `output/` 
folder, completing the ETL pipeline's final **Load** stage. These five 
files together form the "dashboard" — a small, focused set of tables 
answering distinct business questions, ready for further reporting or 
visualization.

In [39]:
import os
os.makedirs("output", exist_ok=True)

view_headcount.toPandas().to_csv("output/headcount_by_department.csv", index=False)
view_avg_salary.toPandas().to_csv("output/avg_salary_by_dept_seniority.csv", index=False)
view_status.toPandas().to_csv("output/status_breakdown_by_department.csv", index=False)
view_top_dept.toPandas().to_csv("output/top_paying_department.csv", index=False)
view_seniority_dist.toPandas().to_csv("output/seniority_distribution.csv", index=False)

## Conclusion

This project demonstrated a complete Spark-based ETL pipeline: extracting 
raw employee data, cleaning it (removing duplicates, handling missing 
values), enriching it with a derived seniority feature, and producing 
five distinct aggregated views that together summarize the organization's 
workforce composition, pay structure, and employment status — mirroring 
how real dashboard-feeding pipelines are built in industry using Apache Spark.